<a href="https://colab.research.google.com/github/tamimrafiUEL/AgriVision-AI/blob/task1-development/Task1_Image_Classification/notebook/Task1_ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AgriVisionAI/Task1"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder created:", PROJECT_DIR)

Project folder created: /content/drive/MyDrive/AgriVisionAI/Task1


In [4]:
!pip install -q kaggle

In [5]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [6]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle authentication file configured.")

Kaggle authentication file configured.


In [7]:
import os

DATA_DOWNLOAD_DIR = "/content/plantvillage_download"

os.makedirs(DATA_DOWNLOAD_DIR, exist_ok=True)

In [8]:
!kaggle datasets download \
    -d mohitsingh1804/plantvillage \
    -p /content/plantvillage_download \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/mohitsingh1804/plantvillage
License(s): GPL-2.0
100% 818M/818M [00:11<00:00, 75.2MB/s]



In [9]:
for root, dirs, files_list in os.walk(DATA_DOWNLOAD_DIR):
    level = root.replace(DATA_DOWNLOAD_DIR, "").count(os.sep)

    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

        for file in files_list[:5]:
            print("  " * (level + 1) + file)

plantvillage_download/
  PlantVillage/
    val/
    train/


In [10]:
import os

for root, dirs, files_list in os.walk(DATA_DOWNLOAD_DIR):
    image_files = [
        f for f in files_list
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    if image_files:
        print(
            "Folder:", root,
            "| Images directly inside:", len(image_files)
        )

Folder: /content/plantvillage_download/PlantVillage/val/Cherry_(including_sour)___healthy | Images directly inside: 170
Folder: /content/plantvillage_download/PlantVillage/val/Raspberry___healthy | Images directly inside: 74
Folder: /content/plantvillage_download/PlantVillage/val/Tomato___Tomato_Yellow_Leaf_Curl_Virus | Images directly inside: 1071
Folder: /content/plantvillage_download/PlantVillage/val/Tomato___Target_Spot | Images directly inside: 281
Folder: /content/plantvillage_download/PlantVillage/val/Corn_(maize)___Common_rust_ | Images directly inside: 239
Folder: /content/plantvillage_download/PlantVillage/val/Potato___Early_blight | Images directly inside: 200
Folder: /content/plantvillage_download/PlantVillage/val/Tomato___Early_blight | Images directly inside: 200
Folder: /content/plantvillage_download/PlantVillage/val/Cherry_(including_sour)___Powdery_mildew | Images directly inside: 210
Folder: /content/plantvillage_download/PlantVillage/val/Corn_(maize)___Cercospora_lea

In [11]:
DATA_DIR = "/content/plantvillage_download/PlantVillage"

print("Dataset folder:", DATA_DIR)
print("Folder exists:", os.path.exists(DATA_DIR))

Dataset folder: /content/plantvillage_download/PlantVillage
Folder exists: True


In [12]:
from pathlib import Path

DATA_DIR = Path(DATA_DIR)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

class_counts = {}

for class_folder in sorted(DATA_DIR.iterdir()):

    if class_folder.is_dir():

        images = [
            p for p in class_folder.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if len(images) > 0:
            class_counts[class_folder.name] = len(images)

print("Number of classes:", len(class_counts))
print("Total number of images:", sum(class_counts.values()))

for class_name, count in class_counts.items():
    print(class_name, ":", count)

Number of classes: 2
Total number of images: 54305
train : 43444
val : 10861


In [13]:
import os

print("Folders inside DATA_DIR:\n")

for item in os.listdir(DATA_DIR):
    item_path = os.path.join(DATA_DIR, item)

    if os.path.isdir(item_path):
        print(f"\n{item}/")

        subfolders = [
            folder for folder in os.listdir(item_path)
            if os.path.isdir(os.path.join(item_path, folder))
        ]

        print("Number of subfolders:", len(subfolders))
        print("First 10 subfolders:", subfolders[:10])

Folders inside DATA_DIR:


val/
Number of subfolders: 38
First 10 subfolders: ['Cherry_(including_sour)___healthy', 'Raspberry___healthy', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Target_Spot', 'Corn_(maize)___Common_rust_', 'Potato___Early_blight', 'Tomato___Early_blight', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Peach___Bacterial_spot']

train/
Number of subfolders: 38
First 10 subfolders: ['Cherry_(including_sour)___healthy', 'Raspberry___healthy', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Target_Spot', 'Corn_(maize)___Common_rust_', 'Potato___Early_blight', 'Tomato___Early_blight', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Peach___Bacterial_spot']


In [14]:
import os

# Main dataset directory
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

# Check folders exist
print("Training folder exists:", os.path.exists(TRAIN_DIR))
print("Validation folder exists:", os.path.exists(VAL_DIR))

Training folder exists: True
Validation folder exists: True


In [15]:
class_names = sorted([
    folder for folder in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, folder))
])

num_classes = len(class_names)

print("Number of disease classes:", num_classes)

for i, name in enumerate(class_names):
    print(i, name)

Number of disease classes: 38
0 Apple___Apple_scab
1 Apple___Black_rot
2 Apple___Cedar_apple_rust
3 Apple___healthy
4 Blueberry___healthy
5 Cherry_(including_sour)___Powdery_mildew
6 Cherry_(including_sour)___healthy
7 Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8 Corn_(maize)___Common_rust_
9 Corn_(maize)___Northern_Leaf_Blight
10 Corn_(maize)___healthy
11 Grape___Black_rot
12 Grape___Esca_(Black_Measles)
13 Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 Grape___healthy
15 Orange___Haunglongbing_(Citrus_greening)
16 Peach___Bacterial_spot
17 Peach___healthy
18 Pepper,_bell___Bacterial_spot
19 Pepper,_bell___healthy
20 Potato___Early_blight
21 Potato___Late_blight
22 Potato___healthy
23 Raspberry___healthy
24 Soybean___healthy
25 Squash___Powdery_mildew
26 Strawberry___Leaf_scorch
27 Strawberry___healthy
28 Tomato___Bacterial_spot
29 Tomato___Early_blight
30 Tomato___Late_blight
31 Tomato___Leaf_Mold
32 Tomato___Septoria_leaf_spot
33 Tomato___Spider_mites Two-spotted_spider_mite


In [16]:
from collections import Counter

train_counts = Counter()
val_counts = Counter()

for class_name in class_names:

    train_path = os.path.join(TRAIN_DIR, class_name)
    val_path = os.path.join(VAL_DIR, class_name)

    train_counts[class_name] = len([
        f for f in os.listdir(train_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    val_counts[class_name] = len([
        f for f in os.listdir(val_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

print("Training images:", sum(train_counts.values()))
print("Validation images:", sum(val_counts.values()))
print("Total images:", sum(train_counts.values()) + sum(val_counts.values()))

Training images: 43444
Validation images: 10861
Total images: 54305


In [17]:
import pandas as pd

class_summary = pd.DataFrame({
    "Disease Class": class_names,
    "Training Images": [train_counts[c] for c in class_names],
    "Validation Images": [val_counts[c] for c in class_names]
})

class_summary["Total Images"] = (
    class_summary["Training Images"] +
    class_summary["Validation Images"]
)

display(class_summary)

print("\nTotal classes:", len(class_summary))
print("Total images:", class_summary["Total Images"].sum())

,Disease Class,Training Images,Validation Images,Total Images
0,Apple___Apple_scab,504,126,630
1,Apple___Black_rot,496,125,621
2,Apple___Cedar_apple_rust,220,55,275
3,Apple___healthy,1316,329,1645
4,Blueberry___healthy,1202,300,1502
5,Cherry_(including_sour)___Powdery_mildew,842,210,1052
6,Cherry_(including_sour)___healthy,684,170,854
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,410,103,513
8,Corn_(maize)___Common_rust_,953,239,1192
9,Corn_(maize)___Northern_Leaf_Blight,788,197,985



Total classes: 38
Total images: 54305


In [18]:
from sklearn.model_selection import train_test_split

# Prepare image paths and labels
image_paths = []
image_labels = []

for class_name in class_names:

    class_path = os.path.join(TRAIN_DIR, class_name)

    for filename in os.listdir(class_path):

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            image_paths.append(
                os.path.join(class_path, filename)
            )

            image_labels.append(class_name)

# Split original training data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    image_paths,
    image_labels,
    test_size=0.20,
    random_state=42,
    stratify=image_labels
)

print("TRAINING IMAGES:", len(X_train))
print("TESTING IMAGES:", len(X_test))

TRAINING IMAGES: 34755
TESTING IMAGES: 8689


In [19]:
X_val = []
y_val = []

for class_name in class_names:

    class_path = os.path.join(VAL_DIR, class_name)

    for filename in os.listdir(class_path):

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            X_val.append(
                os.path.join(class_path, filename)
            )

            y_val.append(class_name)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))

print("\nTotal images:", len(X_train) + len(X_val) + len(X_test))

Training images: 34755
Validation images: 10861
Testing images: 8689

Total images: 54305


In [20]:
print("Training classes:", len(set(y_train)))
print("Validation classes:", len(set(y_val)))
print("Testing classes:", len(set(y_test)))

assert len(set(y_train)) == 38
assert len(set(y_val)) == 38
assert len(set(y_test)) == 38

print("\nSUCCESS: All 38 classes are present in all three datasets!")

Training classes: 38
Validation classes: 38
Testing classes: 38

SUCCESS: All 38 classes are present in all three datasets!
